In [8]:
import os
import pandas as pd
import numpy as np

BASE_DIR   = '/storage/Arushi/090526_EvoAge/kg_formation/'
PROC_DIR   = BASE_DIR + 'processed_data/'
DB_DIR     = BASE_DIR + 'data_collection/databases_for_mapping/'

OUT_PATH   = BASE_DIR + 'processed_data_relation_wise_merge/generalised/PMID_TISSUE/ALL_PMID_TISSUE.parquet'

REQUIRED_COLS = [
    'head', 'relation', 'tail',
    'head_type', 'relation_type', 'tail_type',
    'kg_source',
    'head_id_is', 'tail_id_is',
    'head_detail_name', 'tail_detail_name', 'kg_type', 'species'
]

## 2. Load KG Sources

#### CKG

In [6]:
CKG = pd.read_csv(PROC_DIR + 'CKG/File_42_PMID_Tissue_CKG.csv')
CKG

,Head,Relation,Tail,Head_type,Tail_type,KG_Source,Tail_detail_name,Head_ID_IS,Tail_ID_IS
0,PMID:20174579,MENTIONED_IN_PUBLICATION,BTO:0000003,PMID,Tissue,CKG,intestinal cell line,PubmedID,BrendaTO
1,PMID:20100909,MENTIONED_IN_PUBLICATION,BTO:0000003,PMID,Tissue,CKG,intestinal cell line,PubmedID,BrendaTO
2,PMID:19597572,MENTIONED_IN_PUBLICATION,BTO:0000003,PMID,Tissue,CKG,intestinal cell line,PubmedID,BrendaTO
3,PMID:19597582,MENTIONED_IN_PUBLICATION,BTO:0000003,PMID,Tissue,CKG,intestinal cell line,PubmedID,BrendaTO
4,PMID:19357781,MENTIONED_IN_PUBLICATION,BTO:0000003,PMID,Tissue,CKG,intestinal cell line,PubmedID,BrendaTO
...,...,...,...,...,...,...,...,...,...
303745853,PMID:18566133,MENTIONED_IN_PUBLICATION,BTO:0006321,PMID,Tissue,CKG,GC-1 spg cell,PubmedID,BrendaTO
303745854,PMID:16395525,MENTIONED_IN_PUBLICATION,BTO:0006321,PMID,Tissue,CKG,GC-1 spg cell,PubmedID,BrendaTO
303745855,PMID:15991248,MENTIONED_IN_PUBLICATION,BTO:0006321,PMID,Tissue,CKG,GC-1 spg cell,PubmedID,BrendaTO
303745856,PMID:11179488,MENTIONED_IN_PUBLICATION,BTO:0006321,PMID,Tissue,CKG,GC-1 spg cell,PubmedID,BrendaTO


In [7]:
CKG.columns = CKG.columns.str.lower()
CKG.rename(columns={'tail_go_name': 'tail_detail_name'}, inplace=True)
CKG['relation'] = 'PMID_Tissue'
CKG['tail_type'] = 'Tissue'
CKG['kg_type'] = 'Generalised'
CKG['kg_source'] = 'CKG'
CKG['species'] = 'Homo species'
CKG

,head,relation,tail,head_type,tail_type,kg_source,tail_detail_name,head_id_is,tail_id_is,kg_type,species
0,PMID:20174579,PMID_Tissue,BTO:0000003,PMID,Tissue,CKG,intestinal cell line,PubmedID,BrendaTO,Generalised,Homo species
1,PMID:20100909,PMID_Tissue,BTO:0000003,PMID,Tissue,CKG,intestinal cell line,PubmedID,BrendaTO,Generalised,Homo species
2,PMID:19597572,PMID_Tissue,BTO:0000003,PMID,Tissue,CKG,intestinal cell line,PubmedID,BrendaTO,Generalised,Homo species
3,PMID:19597582,PMID_Tissue,BTO:0000003,PMID,Tissue,CKG,intestinal cell line,PubmedID,BrendaTO,Generalised,Homo species
4,PMID:19357781,PMID_Tissue,BTO:0000003,PMID,Tissue,CKG,intestinal cell line,PubmedID,BrendaTO,Generalised,Homo species
...,...,...,...,...,...,...,...,...,...,...,...
303745853,PMID:18566133,PMID_Tissue,BTO:0006321,PMID,Tissue,CKG,GC-1 spg cell,PubmedID,BrendaTO,Generalised,Homo species
303745854,PMID:16395525,PMID_Tissue,BTO:0006321,PMID,Tissue,CKG,GC-1 spg cell,PubmedID,BrendaTO,Generalised,Homo species
303745855,PMID:15991248,PMID_Tissue,BTO:0006321,PMID,Tissue,CKG,GC-1 spg cell,PubmedID,BrendaTO,Generalised,Homo species
303745856,PMID:11179488,PMID_Tissue,BTO:0006321,PMID,Tissue,CKG,GC-1 spg cell,PubmedID,BrendaTO,Generalised,Homo species


In [9]:
len(set(CKG['head']))

20490325

In [10]:
SOURCE_DFS = [
              ('CKG', CKG)
             ]

for name, df in SOURCE_DFS:
    dupes = df.columns[df.columns.duplicated(keep=False)].unique().tolist()
    print(f"[{name}] {'duplicate columns: ' + str(dupes) if dupes else '✓ no duplicates'}")

[CKG] ✓ no duplicates


In [11]:
CONCAT_COLS = [
    'head', 'relation', 'tail',
    'head_type', 'relation_type', 'tail_type',
    'kg_source', 'head_id_is', 'tail_id_is',
    'head_detail_name', 'tail_detail_name','kg_type', 'species'
]

df_list = []
for name, df in SOURCE_DFS:
    tmp = df.loc[:, ~df.columns.duplicated()].copy()
    for col in CONCAT_COLS:
        if col not in tmp.columns:
            tmp[col] = pd.NA
    df_list.append(tmp[CONCAT_COLS])

final_df = pd.concat(df_list, ignore_index=True)
print(f"Combined: {len(final_df):,} rows")
final_df

Combined: 303,745,858 rows


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_detail_name,kg_type,species
0,PMID:20174579,PMID_Tissue,BTO:0000003,PMID,<NA>,Tissue,CKG,PubmedID,BrendaTO,<NA>,intestinal cell line,Generalised,Homo species
1,PMID:20100909,PMID_Tissue,BTO:0000003,PMID,<NA>,Tissue,CKG,PubmedID,BrendaTO,<NA>,intestinal cell line,Generalised,Homo species
2,PMID:19597572,PMID_Tissue,BTO:0000003,PMID,<NA>,Tissue,CKG,PubmedID,BrendaTO,<NA>,intestinal cell line,Generalised,Homo species
3,PMID:19597582,PMID_Tissue,BTO:0000003,PMID,<NA>,Tissue,CKG,PubmedID,BrendaTO,<NA>,intestinal cell line,Generalised,Homo species
4,PMID:19357781,PMID_Tissue,BTO:0000003,PMID,<NA>,Tissue,CKG,PubmedID,BrendaTO,<NA>,intestinal cell line,Generalised,Homo species
...,...,...,...,...,...,...,...,...,...,...,...,...,...
303745853,PMID:18566133,PMID_Tissue,BTO:0006321,PMID,<NA>,Tissue,CKG,PubmedID,BrendaTO,<NA>,GC-1 spg cell,Generalised,Homo species
303745854,PMID:16395525,PMID_Tissue,BTO:0006321,PMID,<NA>,Tissue,CKG,PubmedID,BrendaTO,<NA>,GC-1 spg cell,Generalised,Homo species
303745855,PMID:15991248,PMID_Tissue,BTO:0006321,PMID,<NA>,Tissue,CKG,PubmedID,BrendaTO,<NA>,GC-1 spg cell,Generalised,Homo species
303745856,PMID:11179488,PMID_Tissue,BTO:0006321,PMID,<NA>,Tissue,CKG,PubmedID,BrendaTO,<NA>,GC-1 spg cell,Generalised,Homo species


In [13]:
final_df['tail_id_is'] = 'BTO_ID'
final_df = final_df[~final_df['tail_detail_name'].isna()]
final_df

,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_detail_name,kg_type,species
0,PMID:20174579,PMID_Tissue,BTO:0000003,PMID,<NA>,Tissue,CKG,PubmedID,BTO_ID,<NA>,intestinal cell line,Generalised,Homo species
1,PMID:20100909,PMID_Tissue,BTO:0000003,PMID,<NA>,Tissue,CKG,PubmedID,BTO_ID,<NA>,intestinal cell line,Generalised,Homo species
2,PMID:19597572,PMID_Tissue,BTO:0000003,PMID,<NA>,Tissue,CKG,PubmedID,BTO_ID,<NA>,intestinal cell line,Generalised,Homo species
3,PMID:19597582,PMID_Tissue,BTO:0000003,PMID,<NA>,Tissue,CKG,PubmedID,BTO_ID,<NA>,intestinal cell line,Generalised,Homo species
4,PMID:19357781,PMID_Tissue,BTO:0000003,PMID,<NA>,Tissue,CKG,PubmedID,BTO_ID,<NA>,intestinal cell line,Generalised,Homo species
...,...,...,...,...,...,...,...,...,...,...,...,...,...
303745853,PMID:18566133,PMID_Tissue,BTO:0006321,PMID,<NA>,Tissue,CKG,PubmedID,BTO_ID,<NA>,GC-1 spg cell,Generalised,Homo species
303745854,PMID:16395525,PMID_Tissue,BTO:0006321,PMID,<NA>,Tissue,CKG,PubmedID,BTO_ID,<NA>,GC-1 spg cell,Generalised,Homo species
303745855,PMID:15991248,PMID_Tissue,BTO:0006321,PMID,<NA>,Tissue,CKG,PubmedID,BTO_ID,<NA>,GC-1 spg cell,Generalised,Homo species
303745856,PMID:11179488,PMID_Tissue,BTO:0006321,PMID,<NA>,Tissue,CKG,PubmedID,BTO_ID,<NA>,GC-1 spg cell,Generalised,Homo species


In [14]:
import pandas as pd
import numpy as np

# Vectorized hash using numpy — avoids Python string ops entirely
keys = (
    pd.util.hash_array(final_df['head'].to_numpy()) ^
    pd.util.hash_array(final_df['relation'].to_numpy()) * 2654435761 ^
    pd.util.hash_array(final_df['tail'].to_numpy()) * 40503
)

# Keep first occurrence of each hash
mask = ~pd.Series(keys).duplicated(keep='first').to_numpy()
final_df = final_df.iloc[mask].reset_index(drop=True)

print(f"After dedup: {len(final_df):,} rows")
final_df

After dedup: 303,745,858 rows


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_detail_name,kg_type,species
0,PMID:20174579,PMID_Tissue,BTO:0000003,PMID,<NA>,Tissue,CKG,PubmedID,BTO_ID,<NA>,intestinal cell line,Generalised,Homo species
1,PMID:20100909,PMID_Tissue,BTO:0000003,PMID,<NA>,Tissue,CKG,PubmedID,BTO_ID,<NA>,intestinal cell line,Generalised,Homo species
2,PMID:19597572,PMID_Tissue,BTO:0000003,PMID,<NA>,Tissue,CKG,PubmedID,BTO_ID,<NA>,intestinal cell line,Generalised,Homo species
3,PMID:19597582,PMID_Tissue,BTO:0000003,PMID,<NA>,Tissue,CKG,PubmedID,BTO_ID,<NA>,intestinal cell line,Generalised,Homo species
4,PMID:19357781,PMID_Tissue,BTO:0000003,PMID,<NA>,Tissue,CKG,PubmedID,BTO_ID,<NA>,intestinal cell line,Generalised,Homo species
...,...,...,...,...,...,...,...,...,...,...,...,...,...
303745853,PMID:18566133,PMID_Tissue,BTO:0006321,PMID,<NA>,Tissue,CKG,PubmedID,BTO_ID,<NA>,GC-1 spg cell,Generalised,Homo species
303745854,PMID:16395525,PMID_Tissue,BTO:0006321,PMID,<NA>,Tissue,CKG,PubmedID,BTO_ID,<NA>,GC-1 spg cell,Generalised,Homo species
303745855,PMID:15991248,PMID_Tissue,BTO:0006321,PMID,<NA>,Tissue,CKG,PubmedID,BTO_ID,<NA>,GC-1 spg cell,Generalised,Homo species
303745856,PMID:11179488,PMID_Tissue,BTO:0006321,PMID,<NA>,Tissue,CKG,PubmedID,BTO_ID,<NA>,GC-1 spg cell,Generalised,Homo species


In [17]:
os.makedirs(os.path.dirname(OUT_PATH), exist_ok=True)
final_df.to_parquet(OUT_PATH, index=False)
print(f"Saved {len(final_df):,} rows → {OUT_PATH}")

Saved 303,745,858 rows → /storage/Arushi/090526_EvoAge/kg_formation/processed_data_relation_wise_merge/generalised/PMID_TISSUE/ALL_PMID_TISSUE.parquet


In [19]:
# PMID_TISSUE